In [1]:
import $ivy.`org.apache.spark::spark-sql:4.1.1`
import org.apache.spark.sql.SparkSession

val spark = SparkSession.builder()
  .appName("Spark411-Almond")
  .master("local[*]")
  .getOrCreate()

val sc = spark.sparkContext

println(s"Java: ${System.getProperty("java.version")}")
println(s"Spark: ${spark.version}")
println(s"Scala: ${scala.util.Properties.versionNumberString}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/27 09:22:31 INFO SparkContext: Running Spark version 4.1.1
26/04/27 09:22:32 INFO SparkContext: OS info Windows 11, 10.0, amd64
26/04/27 09:22:32 INFO SparkContext: Java version 17.0.12+8-LTS-286
26/04/27 09:22:32 INFO ResourceUtils: ==============================================================
26/04/27 09:22:32 INFO ResourceUtils: No custom resources configured for spark.driver.
26/04/27 09:22:32 INFO ResourceUtils: ==============================================================
26/04/27 09:22:32 INFO SparkContext: Submitted application: Spark411-Almond
26/04/27 09:22:32 INFO SecurityManager: Changing view acls to: Imp_06 - Ma26/04/27 09:22:32 INFO SecurityManager: Changing view acls to: Imp_06 - Ma26/04/27 09:22:32 INFO SecurityManager: Changing view acls to: Imp_06 - Ma26/04/27 09:22:32 INFO SecurityManager: Changing view acls to: Imp_06 - Ma26/04/27 09:22:32 INFO SecurityManager: Changing view ac

2026-04-27T07:22:32.704008800Z scala-kernel-interpreter-1 ERROR An exception occurred processing Appender console org.apache.logging.log4j.core.appender.AppenderLoggingException: java.lang.AssertionError: assertion failed
	at org.apache.logging.log4j.core.config.AppenderControl.tryCallAppender(AppenderControl.java:164)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender0(AppenderControl.java:133)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppenderPreventRecursion(AppenderControl.java:124)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender(AppenderControl.java:88)
	at org.apache.logging.log4j.core.config.LoggerConfig.callAppenders(LoggerConfig.java:714)
	at org.apache.logging.log4j.core.config.LoggerConfig.processLogEvent(LoggerConfig.java:672)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:648)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:584)
	at org.apache.logging.log4j.

import $ivy.$
import org.apache.spark.sql.SparkSession
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@4733c3ca
sc: org.apache.spark.SparkContext = org.apache.spark.SparkContext@31ecf689

Ejemplo ampliado de: combineByKey

In [4]:

// RDD de ventas como cadenas: "producto,región,importe"
val ventas = sc.parallelize(List(
  "Laptop,Norte,1200",
  "Teclado,Sur,45",
  "Monitor,Norte,350",
  "Laptop,Sur,1200",
  "Ratón,Norte,25",
  "Monitor,Sur,350",
  "Teclado,Norte,45"
))

// Convertimos a Pair RDD: (región, importe)
val ventasPorRegion = ventas.map { linea =>
  val campos = linea.split(",")
  val region  = campos(1)
  val importe = campos(2).toDouble
  (region, importe)   // ← tupla (clave, valor)
}

ventasPorRegion.collect().foreach(println)
// (Norte,1200.0)
// (Sur,45.0)
// (Norte,350.0)
// (Sur,1200.0)
// (Norte,25.0)
// (Sur,350.0)
// (Norte,45.0)

// Objetivo: para cada región → List con todos los importes
val importesPorRegion = ventasPorRegion.combineByKey(
  (v: Double) => List(v),                       // createCombiner: primer valor → List
  (acc: List[Double], v: Double) => acc :+ v,   // mergeValue: añadir al List
  (acc1: List[Double], acc2: List[Double]) => acc1 ++ acc2 // mergeCombiners: unir Lists 
)

importesPorRegion.collect().foreach { case (region, lista) =>
  println(s"$region → $lista")
}
// Norte → List(1200.0, 350.0, 25.0, 45.0)
// Sur   → List(45.0, 1200.0, 350.0)

2026-04-27T07:30:06.140066700Z scala-kernel-interpreter-1 ERROR An exception occurred processing Appender console org.apache.logging.log4j.core.appender.AppenderLoggingException: java.lang.AssertionError: assertion failed
	at org.apache.logging.log4j.core.config.AppenderControl.tryCallAppender(AppenderControl.java:164)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender0(AppenderControl.java:133)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppenderPreventRecursion(AppenderControl.java:124)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender(AppenderControl.java:88)
	at org.apache.logging.log4j.core.config.LoggerConfig.callAppenders(LoggerConfig.java:714)
	at org.apache.logging.log4j.core.config.LoggerConfig.processLogEvent(LoggerConfig.java:672)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:648)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:584)
	at org.apache.logging.log4j.

ventas: org.apache.spark.rdd.RDD[String] = ParallelCollectionRDD[12] at parallelize at cmd4.sc:3
ventasPorRegion: org.apache.spark.rdd.RDD[(String, Double)] = MapPartitionsRDD[13] at map at cmd4.sc:14
importesPorRegion: org.apache.spark.rdd.RDD[(String, List[Double])] = ShuffledRDD[14] at combineByKey at cmd4.sc:34

 Ejercicio 1: Temperaturas por ciudad

In [5]:
val temperaturas = sc.parallelize(List(
  ("Madrid", 18.5),
  ("Barcelona", 20.0),
  ("Madrid", 21.0),
  ("Valencia", 22.5),
  ("Sevilla", 25.0),
  ("Barcelona", 19.5),
  ("Madrid", 17.0),
  ("Valencia", 23.0),
  ("Sevilla", 26.5),
  ("Barcelona", 18.0),
  ("Madrid", 20.5),
  ("Valencia", 21.5)
), 3)

//verParticiones(temperaturas)

temperaturas: org.apache.spark.rdd.RDD[(String, Double)] = ParallelCollectionRDD[15] at parallelize at cmd5.sc:1

In [7]:
val temperaturaPorRegion = temperaturas.combineByKey(
  (v: Double) => List(v),                       // createCombiner: primer valor → List
  (acc: List[Double], v: Double) => acc :+ v,   // mergeValue: añadir al List
  (acc1: List[Double], acc2: List[Double]) => acc1 ++ acc2 // mergeCombiners: unir Lists 
)

temperaturaPorRegion.collect().foreach { case (region, lista) =>
  println(s"$region → $lista")
}

2026-04-27T07:57:38.343766100Z scala-kernel-interpreter-1 ERROR An exception occurred processing Appender console org.apache.logging.log4j.core.appender.AppenderLoggingException: java.lang.AssertionError: assertion failed
	at org.apache.logging.log4j.core.config.AppenderControl.tryCallAppender(AppenderControl.java:164)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender0(AppenderControl.java:133)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppenderPreventRecursion(AppenderControl.java:124)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender(AppenderControl.java:88)
	at org.apache.logging.log4j.core.config.LoggerConfig.callAppenders(LoggerConfig.java:714)
	at org.apache.logging.log4j.core.config.LoggerConfig.processLogEvent(LoggerConfig.java:672)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:648)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:584)
	at org.apache.logging.log4j.

temperaturaPorRegion: org.apache.spark.rdd.RDD[(String, List[Double])] = ShuffledRDD[17] at combineByKey at cmd7.sc:4

🧪 Ejercicio 2: Productos comprados por cliente

In [8]:
val compras = sc.parallelize(List(
  ("Ana", "Laptop"),
  ("Luis", "Teclado"),
  ("Ana", "Mouse"),
  ("Marta", "Monitor"),
  ("Luis", "Tablet"),
  ("Ana", "Auriculares"),
  ("Carlos", "Impresora"),
  ("Marta", "Webcam"),
  ("Luis", "Ratón"),
  ("Carlos", "SSD"),
  ("Ana", "USB"),
  ("Marta", "Silla")
), 3)


compras: org.apache.spark.rdd.RDD[(String, String)] = ParallelCollectionRDD[18] at parallelize at cmd8.sc:1

In [9]:
val productoPorCliente = compras.combineByKey(
  (v: String) => List(v),                       // createCombiner: primer valor → List
  (acc: List[String], v: String) => acc :+ v,   // mergeValue: añadir al List
  (acc1: List[String], acc2: List[String]) => acc1 ++ acc2 // mergeCombiners: unir Lists 
)

productoPorCliente.collect().foreach { case (region, lista) =>
  println(s"$region → $lista")
}

2026-04-27T08:25:35.598820500Z scala-kernel-interpreter-1 ERROR An exception occurred processing Appender console org.apache.logging.log4j.core.appender.AppenderLoggingException: java.lang.AssertionError: assertion failed
	at org.apache.logging.log4j.core.config.AppenderControl.tryCallAppender(AppenderControl.java:164)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender0(AppenderControl.java:133)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppenderPreventRecursion(AppenderControl.java:124)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender(AppenderControl.java:88)
	at org.apache.logging.log4j.core.config.LoggerConfig.callAppenders(LoggerConfig.java:714)
	at org.apache.logging.log4j.core.config.LoggerConfig.processLogEvent(LoggerConfig.java:672)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:648)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:584)
	at org.apache.logging.log4j.

productoPorCliente: org.apache.spark.rdd.RDD[(String, List[String])] = ShuffledRDD[19] at combineByKey at cmd9.sc:4

🧪 Ejercicio 3: Notas por estudiante

In [11]:
val notas = sc.parallelize(List(
  ("Carlos", 8.5),
  ("Marta", 7.0),
  ("Carlos", 9.0),
  ("Ana", 6.5),
  ("Marta", 8.0),
  ("Ana", 7.5),
  ("Luis", 5.5),
  ("Carlos", 10.0),
  ("Luis", 6.0),
  ("Marta", 9.0),
  ("Ana", 8.5),
  ("Luis", 7.0)
), 3)

notas: org.apache.spark.rdd.RDD[(String, Double)] = ParallelCollectionRDD[21] at parallelize at cmd11.sc:1

In [12]:
val notasPorEstudiante = notas.combineByKey(
  (v: Double) => List(v),                       // createCombiner: primer valor → List
  (acc: List[Double], v: Double) => acc :+ v,   // mergeValue: añadir al List
  (acc1: List[Double], acc2: List[Double]) => acc1 ++ acc2 // mergeCombiners: unir Lists 
)

notasPorEstudiante.collect().foreach { case (region, lista) =>
  println(s"$region → $lista")
}

Marta → List(7.0, 8.0, 9.0)
Carlos → List(8.5, 9.0, 10.0)
Ana → List(6.5, 7.5, 8.5)
Luis → List(5.5, 6.0, 7.0)


notasPorEstudiante: org.apache.spark.rdd.RDD[(String, List[Double])] = ShuffledRDD[22] at combineByKey at cmd12.sc:4

🧪 Ejercicio 4: Palabras por inicial

In [24]:
println(palabras.getNumPartitions)

3


In [14]:
val palabras = sc.parallelize(List(
  ("A", "Azure"),
  ("S", "Spark"),
  ("A", "Almond"),
  ("S", "Scala"),
  ("P", "Python"),
  ("J", "Java"),
  ("A", "Airflow"),
  ("S", "SQL"),
  ("P", "PostgreSQL"),
  ("J", "Jupyter"),
  ("A", "AWS"),
  ("P", "PySpark")
), 3)


palabras: org.apache.spark.rdd.RDD[(String, String)] = ParallelCollectionRDD[24] at parallelize at cmd14.sc:1

In [15]:
val palabrasPorInicial = palabras.combineByKey(
  (v: String) => List(v),                       // createCombiner: primer valor → List
  (acc: List[String], v: String) => acc :+ v,   // mergeValue: añadir al List
  (acc1: List[String], acc2: List[String]) => acc1 ++ acc2 // mergeCombiners: unir Lists 
)

palabrasPorInicial.collect().foreach { case (region, lista) =>
  println(s"$region → $lista")
}

2026-04-27T08:30:53.054086500Z scala-kernel-interpreter-1 ERROR An exception occurred processing Appender console org.apache.logging.log4j.core.appender.AppenderLoggingException: java.lang.AssertionError: assertion failed
	at org.apache.logging.log4j.core.config.AppenderControl.tryCallAppender(AppenderControl.java:164)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender0(AppenderControl.java:133)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppenderPreventRecursion(AppenderControl.java:124)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender(AppenderControl.java:88)
	at org.apache.logging.log4j.core.config.LoggerConfig.callAppenders(LoggerConfig.java:714)
	at org.apache.logging.log4j.core.config.LoggerConfig.processLogEvent(LoggerConfig.java:672)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:648)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:584)
	at org.apache.logging.log4j.

palabrasPorInicial: org.apache.spark.rdd.RDD[(String, List[String])] = ShuffledRDD[25] at combineByKey at cmd15.sc:4

🏢 Contexto empresarial
Una empresa de e-learning (tipo plataforma de cursos online) quiere analizar el comportamiento de sus usuarios. Cada vez que un usuario realiza una acción en la plataforma (ver un vídeo, completar una lección, hacer un test, etc.), se genera un registro. El equipo de Data Engineering necesita preparar los datos para que el equipo de Data Science pueda analizarlos posteriormente.

📥 Datos de entrada
Cada registro tiene la siguiente estructura:

usuario, tipo_evento, duracion_minutos

In [23]:
println(eventos.getNumPartitions)

3


In [16]:
val eventos = sc.parallelize(List(
  ("user1", "video", 10.0),
  ("user2", "quiz", 5.0),
  ("user1", "video", 15.0),
  ("user3", "video", 20.0),
  ("user2", "video", 8.0),
  ("user1", "quiz", 7.0),
  ("user3", "quiz", 6.0),
  ("user2", "video", 12.0),
  ("user1", "video", 9.0),
  ("user3", "video", 11.0),
  ("user2", "quiz", 4.0),
  ("user3", "video", 13.0)
), 3)

eventos: org.apache.spark.rdd.RDD[(String, String, Double)] = ParallelCollectionRDD[26] at parallelize at cmd16.sc:1

In [18]:
// agrupar todas las duraciones de actividad por usuario.
val eventosPorUsuario = eventos.map { case (user, tipo, duracion) =>
  (user, duracion)
}

eventosPorUsuario: org.apache.spark.rdd.RDD[(String, Double)] = MapPartitionsRDD[28] at map at cmd18.sc:2

In [20]:
val tiempoPorEstudiante = eventosPorUsuario.combineByKey(
  (v: Double) => List(v),                       // createCombiner: primer valor → List
  (acc: List[Double], v: Double) => acc :+ v,   // mergeValue: añadir al List
  (acc1: List[Double], acc2: List[Double]) => acc1 ++ acc2 // mergeCombiners: unir Lists 
)

tiempoPorEstudiante.collect().foreach { case (estudiante, lista) =>
  println(s"$estudiante → $lista")
}

2026-04-27T08:37:20.828504800Z scala-kernel-interpreter-1 ERROR An exception occurred processing Appender console org.apache.logging.log4j.core.appender.AppenderLoggingException: java.lang.AssertionError: assertion failed
	at org.apache.logging.log4j.core.config.AppenderControl.tryCallAppender(AppenderControl.java:164)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender0(AppenderControl.java:133)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppenderPreventRecursion(AppenderControl.java:124)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender(AppenderControl.java:88)
	at org.apache.logging.log4j.core.config.LoggerConfig.callAppenders(LoggerConfig.java:714)
	at org.apache.logging.log4j.core.config.LoggerConfig.processLogEvent(LoggerConfig.java:672)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:648)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:584)
	at org.apache.logging.log4j.

tiempoPorEstudiante: org.apache.spark.rdd.RDD[(String, List[Double])] = ShuffledRDD[30] at combineByKey at cmd20.sc:4

In [21]:
//tiempo total por usuario
val sumaPorEstudiante = tiempoPorEstudiante.mapValues(lista => lista.sum)

sumaPorEstudiante.collect().foreach { case (estudiante, total) =>
  println(s"$estudiante → $total")
}


user3 → 50.0
user1 → 41.0
user2 → 29.0


sumaPorEstudiante: org.apache.spark.rdd.RDD[(String, Double)] = MapPartitionsRDD[31] at mapValues at cmd21.sc:2

In [22]:
//tiempo medio por usuario
val mediaPorEstudiante = eventosPorUsuario.combineByKey(
  // 1. createCombiner: Iniciamos la tupla (valor inicial, contador 1)
  (v: Double) => (v, 1),

  // 2. mergeValue: Sumamos el valor al total y sumamos 1 al contador
  (acc: (Double, Int), v: Double) => (acc._1 + v, acc._2 + 1),

  // 3. mergeCombiners: Fusionamos las tuplas de distintos nodos
  (acc1: (Double, Int), acc2: (Double, Int)) => (acc1._1 + acc2._1, acc1._2 + acc2._2)
)

// Finalmente, dividimos la suma entre el contador
mediaPorEstudiante.mapValues { case (suma, contador) => suma / contador }
  .collect()
  .foreach { case (estudiante, promedio) => 
    println(s"Media de $estudiante → $promedio") 
  }


2026-04-27T08:43:43.708367700Z scala-kernel-interpreter-1 ERROR An exception occurred processing Appender console org.apache.logging.log4j.core.appender.AppenderLoggingException: java.lang.AssertionError: assertion failed
	at org.apache.logging.log4j.core.config.AppenderControl.tryCallAppender(AppenderControl.java:164)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender0(AppenderControl.java:133)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppenderPreventRecursion(AppenderControl.java:124)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender(AppenderControl.java:88)
	at org.apache.logging.log4j.core.config.LoggerConfig.callAppenders(LoggerConfig.java:714)
	at org.apache.logging.log4j.core.config.LoggerConfig.processLogEvent(LoggerConfig.java:672)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:648)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:584)
	at org.apache.logging.log4j.

mediaPorEstudiante: org.apache.spark.rdd.RDD[(String, (Double, Int))] = ShuffledRDD[32] at combineByKey at cmd22.sc:10